In [21]:
import os
import re

def parse_mean_std(line: str):
    s = line.replace('Â', '').replace('\xa0', ' ')
    m = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
    if len(m) >= 2:
        return float(m[-2]), float(m[-1]) 
    raise ValueError(f"Could not parse numbers from: {line}")


def parse_results(dataset_directory):
    group_n_teach_average = []
    sort_average = []
    none_average = []

    group_n_teach_std = []
    sort_std = []
    none_std = []

    icl_files = os.listdir(dataset_directory)

    icl_files = sorted(icl_files)

    for icl_file in icl_files:

        current_icl = os.listdir(f'{dataset_directory}/{icl_file}')

        current_icl = sorted(current_icl)

        for method in current_icl:

            with open(f'{dataset_directory}/{icl_file}/{method}/metrics.txt') as f:
                for line in f:
                    if ('RMSE Gemini vs Original' in line):
                        if ('group_n_teach' in method):
                            group_n_teach_average.append(line)
                            mean_val, std_val = parse_mean_std(line)
                            group_n_teach_std.append(std_val)
                            group_n_teach_average[-1] = mean_val

                        elif ('sort' in method):
                            sort_average.append(line)
                            mean_val, std_val = parse_mean_std(line)
                            sort_std.append(std_val)
                            sort_average[-1] = mean_val

                        elif('none' in method):
                            none_average.append(line)
                            mean_val, std_val = parse_mean_std(line)
                            none_std.append(std_val)
                            none_average[-1] = mean_val

    return group_n_teach_average, group_n_teach_std, sort_average, sort_std, none_average, none_std

In [22]:
import matplotlib.pyplot as plt
import numpy as np

def plot_svd_rmse_curves(
    group_means=None,
    sort_means=None,
    none_means=None,
    random_baseline=None,
    numpy_baseline=None,
    title="Rank-1 RMSE vs ICL examples",
    outpath=None
):
    def _as_float_list(x):
        if x is None: return []
        return [float(v) for v in x]

    series = []
    gm = _as_float_list(group_means)
    sm = _as_float_list(sort_means)
    nm = _as_float_list(none_means)

    if len(gm) > 0: series.append(("GroupNTeach", gm, 'o'))
    if len(sm) > 0: series.append(("Sort", sm, 's'))
    if len(nm) > 0: series.append(("None", nm, '^'))

    if len(series) == 0:
        raise ValueError("No permutation series provided (all empty).")

    max_len = max(len(s[1]) for s in series)
    x_ticks = list(range(max_len))

    plt.figure(figsize=(8, 5))
    for label, means, marker in series:
        x = np.arange(len(means))
        y = np.array(means, dtype=float)
        plt.plot(x, y, marker=marker, linewidth=1.8, label=label)

    if random_baseline is not None:
        plt.axhline(float(random_baseline), linestyle='--', color='purple', linewidth=1.2, label='Random baseline')

    if numpy_baseline is not None:
        plt.axhline(float(numpy_baseline), linestyle='-.', color='red', linewidth=1.2, label='NumPy rank-1 baseline')

    plt.title(title)
    plt.xlabel("ICL examples")
    plt.ylabel("RMSE")
    plt.xticks(x_ticks, x_ticks)
    plt.ylim(bottom=0)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()

    if outpath:
        plt.savefig(outpath, dpi=200)
        plt.close()
    else:
        plt.show()


In [23]:
cifar_10_random_rmse_average = 120.979501
cifar_10_numpy_random_rmse = 33.251059

cifar10_group_n_teach_RMSE_average = []
cifar10_group_n_teach_RMSE_std = []

cifar10_sort_RMSE_average = []
cifar10_sort_RMSE_std = []

cifar10_directory = 'Results_Cifar10_32x32'

cifar10_group_n_teach_RMSE_average, cifar10_group_n_teach_RMSE_std, cifar10_sort_RMSE_average, cifar10_sort_RMSE_std, _, _, = parse_results(cifar10_directory)

plot_svd_rmse_curves(
    group_means=cifar10_group_n_teach_RMSE_average,
    sort_means=cifar10_sort_RMSE_average,
    none_means=[], 
    random_baseline=cifar_10_random_rmse_average,
    numpy_baseline=cifar_10_numpy_random_rmse,
    title="CIFAR-10 Rank‑1 RMSE vs. ICL Examples",
    outpath="cifar10_rmse_plot.png"
)



In [24]:
digits_random_rmse_average = 119.591299
digits_numpy_random_rmse = 54.300136

digits_group_n_teach_RMSE_average = []
digits_group_n_teach_RMSE_std = []

digits_sort_RMSE_average = []
digits_sort_RMSE_std = []

digits_none_RMSE_average = []
digits_none_RMSE_std = []

digits_directory = 'Results_Digits_8x8'

digits_group_n_teach_RMSE_average, digits_group_n_teach_RMSE_std, digits_sort_RMSE_average, digits_sort_RMSE_std, digits_none_RMSE_average, digits_none_RMSE_std = parse_results(digits_directory)

plot_svd_rmse_curves(
    group_means=digits_group_n_teach_RMSE_average,
    sort_means=digits_sort_RMSE_average,
    none_means=digits_none_RMSE_average, 
    random_baseline=digits_random_rmse_average,
    numpy_baseline=digits_numpy_random_rmse,
    title="Digits Rank‑1 RMSE vs. ICL Examples",
    outpath="digits_rmse_plot.png"
)

print(digits_group_n_teach_RMSE_average, digits_sort_RMSE_average, digits_none_RMSE_average)

[68.344074, 62.925709, 62.260812, 61.71974, 62.198099, 61.324157] [66.727581, 62.578749, 61.581639, 61.676619, 61.25199, 61.214312] [70.399451, 65.25503, 64.151859, 63.499569, 63.93228, 63.136393]


In [25]:
digits_random_rmse_average = 119.591299
digits_numpy_random_rmse = 54.300136

digits_group_n_teach_RMSE_average = []
digits_group_n_teach_RMSE_std = []

digits_sort_RMSE_average = []
digits_sort_RMSE_std = []

digits_none_RMSE_average = []
digits_none_RMSE_std = []

digits_directory = 'Results_Digits_8x8'

digits_group_n_teach_RMSE_average, digits_group_n_teach_RMSE_std, digits_sort_RMSE_average, digits_sort_RMSE_std, digits_none_RMSE_average, digits_none_RMSE_std = parse_results(digits_directory)

plot_svd_rmse_curves(
    group_means=digits_group_n_teach_RMSE_average,
    sort_means=digits_sort_RMSE_average,
    none_means=digits_none_RMSE_average, 
    random_baseline=digits_random_rmse_average,
    numpy_baseline=digits_numpy_random_rmse,
    title="Digits Rank‑1 RMSE vs. ICL Examples",
    outpath="digits_rmse_plot.png"
)

In [26]:
synthetic_noisy_random_rmse_average = 86.349158
synthetic_noisy_numpy_random_rmse = 60.609821

synthetic_noisy_group_n_teach_RMSE_average = []
synthetic_noisy_group_n_teach_RMSE_std = []

synthetic_noisy_sort_RMSE_average = []
synthetic_noisy_sort_RMSE_std = []

synthetic_noisy_none_RMSE_average = []
synthetic_noisy_none_RMSE_std = []

synthetic_directory = 'Results_Synthetic_Noisy_16x16' 

synthetic_noisy_group_n_teach_RMSE_average, synthetic_noisy_group_n_teach_RMSE_std, \
synthetic_noisy_sort_RMSE_average, synthetic_noisy_sort_RMSE_std, \
synthetic_noisy_none_RMSE_average, synthetic_noisy_none_RMSE_std = parse_results(synthetic_directory)

plot_svd_rmse_curves(
    group_means=synthetic_noisy_group_n_teach_RMSE_average,
    sort_means=synthetic_noisy_sort_RMSE_average,
    none_means=synthetic_noisy_none_RMSE_average,
    random_baseline=synthetic_noisy_random_rmse_average,  
    numpy_baseline=synthetic_noisy_numpy_random_rmse,     
    title="Synthetic (Noisy) Rank‑1 RMSE vs. ICL Examples",
    outpath="synthetic_noisy_rmse_plot.png"
)


In [27]:
synthetic_random_rmse_average = 86.339753
synthetic_numpy_random_rmse = 60.598492

synthetic_group_n_teach_RMSE_average = []
synthetic_group_n_teach_RMSE_std = []

synthetic_sort_RMSE_average = []
synthetic_sort_RMSE_std = []

synthetic_none_RMSE_average = []
synthetic_none_RMSE_std = []

synthetic_directory = 'Results_Synthetic_16x16' 

synthetic_group_n_teach_RMSE_average, synthetic_group_n_teach_RMSE_std, \
synthetic_sort_RMSE_average, synthetic_sort_RMSE_std, \
synthetic_none_RMSE_average, synthetic_none_RMSE_std = parse_results(synthetic_directory)

plot_svd_rmse_curves(
    group_means=synthetic_group_n_teach_RMSE_average,
    sort_means=synthetic_sort_RMSE_average,
    none_means=synthetic_none_RMSE_average,
    random_baseline=synthetic_random_rmse_average,  
    numpy_baseline=synthetic_numpy_random_rmse,     
    title="Synthetic Rank‑1 RMSE vs. ICL Examples",
    outpath="synthetic_rmse_plot.png"
)
